# 04 - Plan de generación de tiles para grilla temporal S3

Este notebook calcula cuántos tiles Sentinel-2 faltan para construir el tensor temporal pedido por la rúbrica:

```text
N x N x 8 x 256
```

No genera imágenes todavía. Solo define requerimientos y cobertura usando:

- grilla 0.005 grados de Situación 3;
- fechas Sentinel-2 disponibles;
- tiles/embeddings ya existentes de Situación 2;
- estaciones DAGMA como zona prioritaria.


In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

BASE = Path('/workspace/geovision-cali-hf')
OUT = BASE / 'outputs/situacion3/rubrica/04_plan_generacion_tiles_grilla_temporal_s3'
OUT.mkdir(parents=True, exist_ok=True)

PATHS = {
    'grid_s3': BASE / 'outputs/situacion3/02_grilla_s2_features/grid_cali_005deg.parquet',
    'station_to_grid_s3': BASE / 'outputs/situacion3/02_grilla_s2_features/station_to_s2_grid_mapping.csv',
    's2_grid_features': BASE / 'outputs/situacion3/02_grilla_s2_features/s2_grid_features_with_quality.parquet',
    'metadata_s2_v5b': BASE / 'outputs/clip_dataset_final_step_by_step/clip_s2_12band_pdf_classes_v5b_high_purity_1500/metadata.jsonl',
    'embeddings_remoteclip_v5b': BASE / 'outputs/clip_training_remoteclip_v10_v5b_embeddings/embeddings_remoteclip_v10_v5b.npz',
    'checkpoint_sae_v5b': BASE / 'outputs/clip_training_remoteclip_fusion_ksae_v10_v5b_gsplit_seed_sweep/seed57_k12_drop25_wd1e3_best.pt',
}

def md5_file(path, chunk_size=1024 * 1024):
    h = hashlib.md5()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(chunk_size), b''):
            h.update(chunk)
    return h.hexdigest()

pd.DataFrame([
    {
        'name': k,
        'path': str(v),
        'exists': v.exists(),
        'size_mb': round(v.stat().st_size / 1024 / 1024, 3) if v.exists() else None,
        'md5': md5_file(v) if v.exists() and v.is_file() else None,
    }
    for k, v in PATHS.items()
])


,name,path,exists,size_mb,md5
0,grid_s3,/workspace/geovision-cali-hf/outputs/situacion...,True,0.033,cf11e3abf87e99070475a226e363f10b
1,station_to_grid_s3,/workspace/geovision-cali-hf/outputs/situacion...,True,0.001,089f931167b57a9e1d074996844e1cbd
2,s2_grid_features,/workspace/geovision-cali-hf/outputs/situacion...,True,45.546,721bfc1e6443b35c470f5a3afd770c1b
3,metadata_s2_v5b,/workspace/geovision-cali-hf/outputs/clip_data...,True,2.376,5e0a96c7572e00f21edb8e7a864b977d
4,embeddings_remoteclip_v5b,/workspace/geovision-cali-hf/outputs/clip_trai...,True,2.635,6cb3b11bede28d5568b9253a9a5e1d8d
5,checkpoint_sae_v5b,/workspace/geovision-cali-hf/outputs/clip_trai...,True,9.028,2a512b70e4a837a0c652aa71f2c970b4


## Carga base

Se cargan grilla, fechas Sentinel-2 disponibles, estaciones y tiles existentes de Situación 2.

In [2]:
grid = pd.read_parquet(PATHS['grid_s3']).copy()
stations = pd.read_csv(PATHS['station_to_grid_s3'])
s2_features = pd.read_parquet(PATHS['s2_grid_features'], columns=['date', 'grid_id', 'lat', 'lon', 's2_quality_pass'])
meta_s2 = pd.read_json(PATHS['metadata_s2_v5b'], lines=True)
emb_npz = np.load(PATHS['embeddings_remoteclip_v5b'], allow_pickle=True)

s2_features['date'] = pd.to_datetime(s2_features['date']).dt.normalize()
meta_s2['date_day'] = pd.to_datetime(meta_s2['date_day']).dt.normalize()

grid_summary = {
    'n_grid_cells': int(len(grid)),
    'n_s2_feature_rows': int(len(s2_features)),
    'n_s2_dates': int(s2_features['date'].nunique()),
    's2_date_min': str(s2_features['date'].min().date()),
    's2_date_max': str(s2_features['date'].max().date()),
    'n_station_grid_cells': int(stations['nearest_grid_id'].nunique()),
    'n_s2_tiles_existing': int(len(meta_s2)),
    'n_existing_tile_dates': int(meta_s2['date_day'].nunique()),
}

grid_summary


{'n_grid_cells': 2000,
 'n_s2_feature_rows': 226352,
 'n_s2_dates': 129,
 's2_date_min': '2020-01-02',
 's2_date_max': '2024-12-16',
 'n_station_grid_cells': 9,
 'n_s2_tiles_existing': 1500,
 'n_existing_tile_dates': 115}

## Fechas de entrada de 8 pasos

La rúbrica pide las últimas 8 fechas históricas. Para no crear todos los productos posibles de golpe, aquí definimos eventos de predicción usando cada fecha Sentinel-2 desde la octava disponible.

Cada muestra usa:

```text
8 fechas Sentinel-2 previas/incluyendo fecha de corte
```


In [3]:
s2_dates = np.array(sorted(s2_features['date'].unique()))
sequence_length = 8

sample_rows = []
for sample_id, end_idx in enumerate(range(sequence_length - 1, len(s2_dates))):
    seq_dates = [pd.Timestamp(d).date().isoformat() for d in s2_dates[end_idx - sequence_length + 1:end_idx + 1]]
    sample_rows.append({
        'sample_id': sample_id,
        'input_start_date': seq_dates[0],
        'input_end_date': seq_dates[-1],
        'sequence_dates_json': json.dumps(seq_dates),
    })

samples = pd.DataFrame(sample_rows)
sequence_summary = {
    'sequence_length': sequence_length,
    'n_s2_dates': int(len(s2_dates)),
    'n_prediction_samples': int(len(samples)),
    'first_sample': samples.iloc[0].to_dict() if len(samples) else None,
    'last_sample': samples.iloc[-1].to_dict() if len(samples) else None,
}
sequence_summary


{'sequence_length': 8,
 'n_s2_dates': 129,
 'n_prediction_samples': 122,
 'first_sample': {'sample_id': 0,
  'input_start_date': '2020-01-02',
  'input_end_date': '2020-02-11',
  'sequence_dates_json': '["2020-01-02", "2020-01-07", "2020-01-12", "2020-01-17", "2020-01-27", "2020-02-01", "2020-02-06", "2020-02-11"]'},
 'last_sample': {'sample_id': 121,
  'input_start_date': '2024-09-17',
  'input_end_date': '2024-12-16',
  'sequence_dates_json': '["2024-09-17", "2024-10-02", "2024-10-12", "2024-10-17", "2024-11-06", "2024-12-01", "2024-12-11", "2024-12-16"]'}}

## Requerimientos de tiles

Se calcula cuántos tiles serían necesarios para formar tensores completos sobre la grilla Cali 50x40 y también sobre una zona prioritaria alrededor de estaciones DAGMA.

Esto evita empezar generando un volumen innecesario.

In [4]:
def haversine_km(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 6371.0088 * 2 * np.arcsin(np.sqrt(a))

station_cells = set(stations['nearest_grid_id'].astype(str))
station_lat = stations['station_lat'].to_numpy()
station_lon = stations['station_lon'].to_numpy()

priority_rows = []
for row in grid.itertuples(index=False):
    d = haversine_km(float(row.lon), float(row.lat), station_lon, station_lat)
    priority_rows.append({
        'grid_id': row.grid_id,
        'lat': float(row.lat),
        'lon': float(row.lon),
        'min_station_distance_km': float(np.min(d)),
        'is_station_cell': str(row.grid_id) in station_cells,
        'within_1km_station': bool(np.min(d) <= 1.0),
        'within_2km_station': bool(np.min(d) <= 2.0),
        'within_5km_station': bool(np.min(d) <= 5.0),
    })
priority_grid = pd.DataFrame(priority_rows)

priority_summary = {
    'all_grid_cells': int(len(priority_grid)),
    'station_cells': int(priority_grid['is_station_cell'].sum()),
    'within_1km_station_cells': int(priority_grid['within_1km_station'].sum()),
    'within_2km_station_cells': int(priority_grid['within_2km_station'].sum()),
    'within_5km_station_cells': int(priority_grid['within_5km_station'].sum()),
}
priority_summary


{'all_grid_cells': 2000,
 'station_cells': 9,
 'within_1km_station_cells': 81,
 'within_2km_station_cells': 303,
 'within_5km_station_cells': 1020}

## Cobertura de tiles existentes de Situación 2

Mapeamos los 1500 tiles existentes a la grilla para saber qué requerimientos ya están cubiertos.

In [5]:
grid_lat = grid['lat'].to_numpy()
grid_lon = grid['lon'].to_numpy()
grid_ids = grid['grid_id'].astype(str).to_numpy()

existing_rows = []
for row in meta_s2.reset_index(drop=True).itertuples(index=False):
    lat = float(row.s5p_lat_center)
    lon = float(row.s5p_lon_center)
    d = haversine_km(lon, lat, grid_lon, grid_lat)
    j = int(np.argmin(d))
    existing_rows.append({
        'pair_id': row.pair_id,
        'date': pd.Timestamp(row.date_day).date().isoformat(),
        'nearest_grid_id': grid_ids[j],
        'distance_km_to_grid': float(d[j]),
        'image_path': row.image_path,
        'scl_path': row.scl_path,
    })
existing_tiles = pd.DataFrame(existing_rows)
existing_pairs = set(zip(existing_tiles['nearest_grid_id'], existing_tiles['date']))

existing_summary = {
    'n_existing_tiles': int(len(existing_tiles)),
    'n_existing_grid_date_pairs': int(len(existing_pairs)),
    'n_existing_grid_cells': int(existing_tiles['nearest_grid_id'].nunique()),
    'n_existing_dates': int(existing_tiles['date'].nunique()),
    'median_distance_km_to_grid': float(existing_tiles['distance_km_to_grid'].median()),
    'max_distance_km_to_grid': float(existing_tiles['distance_km_to_grid'].max()),
}
existing_summary


{'n_existing_tiles': 1500,
 'n_existing_grid_date_pairs': 1477,
 'n_existing_grid_cells': 594,
 'n_existing_dates': 115,
 'median_distance_km_to_grid': 0.21299660111796487,
 'max_distance_km_to_grid': 0.33413096003202375}

## Tamaño del trabajo requerido

Se estima el número de tiles necesarios en tres escenarios:

- grilla completa;
- celdas a 5 km de estaciones;
- celdas de estación únicamente.


In [6]:
def requirement_summary(name, grid_subset):
    req_pairs = set()
    for gid in grid_subset['grid_id'].astype(str):
        for d in s2_dates:
            req_pairs.add((gid, pd.Timestamp(d).date().isoformat()))
    covered = req_pairs & existing_pairs
    missing = req_pairs - existing_pairs
    return {
        'scenario': name,
        'n_grid_cells': int(len(grid_subset)),
        'n_dates': int(len(s2_dates)),
        'required_grid_date_pairs': int(len(req_pairs)),
        'already_covered_by_s2_existing': int(len(covered)),
        'missing_grid_date_pairs': int(len(missing)),
        'coverage_fraction': float(len(covered) / len(req_pairs)) if req_pairs else 0.0,
    }

requirement_summaries = pd.DataFrame([
    requirement_summary('grilla_completa_cali', priority_grid),
    requirement_summary('zona_5km_estaciones', priority_grid[priority_grid['within_5km_station']]),
    requirement_summary('zona_2km_estaciones', priority_grid[priority_grid['within_2km_station']]),
    requirement_summary('zona_1km_estaciones', priority_grid[priority_grid['within_1km_station']]),
    requirement_summary('solo_celdas_estacion', priority_grid[priority_grid['is_station_cell']]),
])
requirement_summaries


,scenario,n_grid_cells,n_dates,required_grid_date_pairs,already_covered_by_s2_existing,missing_grid_date_pairs,coverage_fraction
0,grilla_completa_cali,2000,129,258000,1442,256558,0.005589
1,zona_5km_estaciones,1020,129,131580,809,130771,0.006148
2,zona_2km_estaciones,303,129,39087,285,38802,0.007291
3,zona_1km_estaciones,81,129,10449,107,10342,0.010240
4,solo_celdas_estacion,9,129,1161,5,1156,0.004307


## Requerimientos faltantes priorizados

Se construye una tabla de faltantes para el escenario inicial recomendado: celdas a 2 km de estaciones. Este escenario reduce costo y permite entrenar/validar contra DAGMA antes de escalar a toda Cali.

In [7]:
scenario_grid = priority_grid[priority_grid['within_2km_station']].copy()
missing_rows = []
for row in scenario_grid.itertuples(index=False):
    for d in s2_dates:
        date_str = pd.Timestamp(d).date().isoformat()
        has_existing = (str(row.grid_id), date_str) in existing_pairs
        missing_rows.append({
            'grid_id': row.grid_id,
            'date': date_str,
            'lat': float(row.lat),
            'lon': float(row.lon),
            'min_station_distance_km': float(row.min_station_distance_km),
            'tile_exists_from_situacion2': bool(has_existing),
            'needs_generation': bool(not has_existing),
        })
missing_plan_2km = pd.DataFrame(missing_rows)

missing_plan_summary = {
    'scenario': 'zona_2km_estaciones',
    'n_rows': int(len(missing_plan_2km)),
    'already_existing': int(missing_plan_2km['tile_exists_from_situacion2'].sum()),
    'needs_generation': int(missing_plan_2km['needs_generation'].sum()),
    'coverage_fraction': float(missing_plan_2km['tile_exists_from_situacion2'].mean()),
}
missing_plan_summary


{'scenario': 'zona_2km_estaciones',
 'n_rows': 39087,
 'already_existing': 285,
 'needs_generation': 38802,
 'coverage_fraction': 0.007291426817100315}

## Guardado

Se guardan los resúmenes para decidir el siguiente notebook: generar tiles faltantes o reducir escenario.

In [8]:
priority_grid_path = OUT / 'priority_grid_station_buffers.csv'
existing_tiles_path = OUT / 'existing_s2_tiles_mapped_to_grid.csv'
requirements_path = OUT / 'tile_generation_requirement_summaries.csv'
missing_plan_path = OUT / 'missing_tile_plan_2km_stations.csv'
manifest_path = OUT / 'manifest_04_plan_generacion_tiles_grilla_temporal_s3.json'

priority_grid.to_csv(priority_grid_path, index=False)
existing_tiles.to_csv(existing_tiles_path, index=False)
requirement_summaries.to_csv(requirements_path, index=False)
missing_plan_2km.to_csv(missing_plan_path, index=False)

manifest = {
    'notebook': '04_plan_generacion_tiles_grilla_temporal_s3.ipynb',
    'description': 'Plan cuantitativo de tiles necesarios para construir embeddings N x N x 8 x 256 alineados al PDF.',
    'inputs': {k: str(v) for k, v in PATHS.items()},
    'input_md5': {k: md5_file(v) for k, v in PATHS.items() if v.exists() and v.is_file()},
    'summaries': {
        'grid_summary': grid_summary,
        'sequence_summary': sequence_summary,
        'priority_summary': priority_summary,
        'existing_summary': existing_summary,
        'missing_plan_2km': missing_plan_summary,
    },
    'outputs': {
        'priority_grid': str(priority_grid_path),
        'existing_tiles': str(existing_tiles_path),
        'requirements': str(requirements_path),
        'missing_plan_2km': str(missing_plan_path),
    },
}
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')

manifest['summaries']


{'grid_summary': {'n_grid_cells': 2000,
  'n_s2_feature_rows': 226352,
  'n_s2_dates': 129,
  's2_date_min': '2020-01-02',
  's2_date_max': '2024-12-16',
  'n_station_grid_cells': 9,
  'n_s2_tiles_existing': 1500,
  'n_existing_tile_dates': 115},
 'sequence_summary': {'sequence_length': 8,
  'n_s2_dates': 129,
  'n_prediction_samples': 122,
  'first_sample': {'sample_id': 0,
   'input_start_date': '2020-01-02',
   'input_end_date': '2020-02-11',
   'sequence_dates_json': '["2020-01-02", "2020-01-07", "2020-01-12", "2020-01-17", "2020-01-27", "2020-02-01", "2020-02-06", "2020-02-11"]'},
  'last_sample': {'sample_id': 121,
   'input_start_date': '2024-09-17',
   'input_end_date': '2024-12-16',
   'sequence_dates_json': '["2024-09-17", "2024-10-02", "2024-10-12", "2024-10-17", "2024-11-06", "2024-12-01", "2024-12-11", "2024-12-16"]'}},
 'priority_summary': {'all_grid_cells': 2000,
  'station_cells': 9,
  'within_1km_station_cells': 81,
  'within_2km_station_cells': 303,
  'within_5km_stat